# Week 03 Adapter Sprint: DPO run on the verified SFT adapter (Colab T4)

**Purpose:** one baseline DPO pass (β = 0.1, LR 5e-5, 1 epoch, 150 frozen preference pairs), starting from the verified baseline SFT adapter.

**Architecture:**
- **Policy:** the SFT adapter, trainable.
- **Reference:** TRL's frozen copy of the *same* SFT adapter (adapter name `ref`) on the same 4-bit base. No merge and no second base model.
- The policy is restored to fp32 after TRL initialization, so it starts **bit-identical** to the SFT adapter. A step-0 check requires all rewards to be exactly 0.

All logic lives in `training/train_dpo.py`. This notebook only calls repository scripts.

**Rules:**
- If any cell fails, **stop**. Do not change settings and re-run. Failed CUDA runs are logged automatically.
- No numbers are pre-filled.
- **Reward accuracy on these 150 pairs is TRAINING-SET reward accuracy**, not held-out generalization.

## 1. Runtime / GPU check (before installing anything)

In [ ]:
!nvidia-smi
import sys, torch
print("python", sys.version.split()[0])
print("preinstalled torch", torch.__version__, "| built for CUDA", torch.version.cuda, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("STOP: no CUDA GPU. Runtime > Change runtime type > T4 GPU, then restart from Section 1.")
print("GPU:", torch.cuda.get_device_name(0), "| capability:", torch.cuda.get_device_capability(0))
with open("/content/torch_constraint.txt", "w") as f:
    f.write(f"torch=={torch.__version__}\n")
print(open("/content/torch_constraint.txt").read())

## 2. Clone the repository (exact reviewed commit required)

In [ ]:
REPO_URL = "https://github.com/brentoatis-collab/week03-adapter-sprint.git"
REPO_DIR = "/content/week03-adapter-sprint"
EXPECTED_COMMIT = ""   # REQUIRED: the reviewed Step 5B commit

import os
if not EXPECTED_COMMIT:
    raise RuntimeError("STOP: set EXPECTED_COMMIT to the reviewed commit.")
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -5

## 3. Install pinned dependencies
torch stays pinned to Colab's preinstalled CUDA build. A `pip check` line saying *ipython requires jedi* comes from Colab's own IPython and needs no action (README §6). Any other issue, especially one naming a pinned package, is a stop condition.

In [ ]:
%cd /content/week03-adapter-sprint
!pip install -r requirements.txt -c /content/torch_constraint.txt
!pip check || echo "pip check reported issues (see above) - review before continuing"

## 4. ⚠️ RESTART REQUIRED
**Runtime → Restart session**, then continue from **Section 5**. Do not re-run Sections 1–3.

## 5. Preflight on the exact commit

In [ ]:
import subprocess

def run(cmd):
    """Run a repository script, stream its output, and STOP the notebook on a non-zero exit."""
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"STOP: command failed with exit code {rc}: {' '.join(cmd)}")

import os, sys
os.chdir("/content/week03-adapter-sprint")
EXPECTED_COMMIT = ""   # REQUIRED again after the restart (same value as Section 2)
if not EXPECTED_COMMIT:
    raise RuntimeError("STOP: set EXPECTED_COMMIT.")
run([sys.executable, "training/preflight.py", "--require-cuda", "--expected-commit", EXPECTED_COMMIT])

## 6. Restore the verified SFT adapter from Google Drive
DPO never assumes an earlier Colab session still exists.
- Copies the SFT run directory from Drive into `outputs/qlora_sft/qlora_sft-baseline-20260927T235802Z/`.
- Checks the adapter SHA-256 against the value recorded in `configs/adapter_config.json` (`dpo.expected_sft_adapter_sha256`) **and** against the logged SFT row.
- Stops on any mismatch.

In [ ]:
from google.colab import drive
import json, shutil, hashlib, csv
drive.mount("/content/drive")
SFT_RUN_ID = "qlora_sft-baseline-20260927T235802Z"
src = f"/content/drive/MyDrive/week03_adapter_sprint/qlora_sft/{SFT_RUN_ID}"
dst = f"outputs/qlora_sft/{SFT_RUN_ID}"
if not os.path.isdir(src):
    raise RuntimeError(f"STOP: SFT run directory not found on Drive: {src}")
if os.path.exists(dst):
    raise RuntimeError(f"STOP: {dst} already exists in this runtime; not overwriting.")
shutil.copytree(src, dst)
cfg = json.load(open("configs/adapter_config.json"))
expected = cfg["dpo"]["expected_sft_adapter_sha256"]
weights = os.path.join(dst, "final_adapter", "adapter_model.safetensors")
actual = hashlib.sha256(open(weights, "rb").read()).hexdigest()
logged = [r for r in csv.DictReader(open("logs/experiment_log.csv")) if r["run_id"] == SFT_RUN_ID]
print("expected (config):", expected)
print("restored file    :", actual)
print("logged SFT row   :", logged[0]["adapter_sha256"] if logged else "MISSING")
if not (actual == expected == (logged[0]["adapter_sha256"] if logged else None)):
    raise RuntimeError("STOP: SFT adapter SHA-256 mismatch.")
SFT_ADAPTER_DIR = os.path.join(dst, "final_adapter")
print("SFT adapter verified:", SFT_ADAPTER_DIR)

## 7. Dataset integrity (frozen instruction + preference data; no regeneration)

In [ ]:
run([sys.executable, "scripts/audit_instruction_data.py"])
run([sys.executable, "scripts/audit_preference_data.py"])
run(["git", "diff", "--exit-code", "--stat", "--", "data/"])
print("data/ matches the committed frozen datasets")

## 8. DPO dry run (strict)
Re-verifies the frozen data hashes and the SFT adapter SHA-256. For every pair it tokenizes the prompt and both completions exactly as TRL does, and **fails** if:
- the prompt is not an exact token prefix,
- a completion does not reproduce its response, or
- any sequence would be truncated at `max_length` 512.

It also prints the step plan.

In [ ]:
run([sys.executable, "training/train_dpo.py", "--sft-adapter-dir", SFT_ADAPTER_DIR, "--dry-run"])

## 9. DPO run (baseline, no overrides)

Expected **10 optimizer steps**:
- 150 pairs ÷ 2 per micro-batch = 75 micro-batches.
- 75 = 9 × 8 + 3, so 9 full accumulation steps plus 1 partial step of 3 micro-batches (6 pairs).
- Warmup is `ceil(0.1 × 10)` = 1 step.

Look in the output for:
- `QUANTIZATION EVIDENCE`
- `POLICY / REFERENCE SETUP`: policy == ref == SFT file, fp32
- `STEP-0 REFERENCE CHECK`: rewards must be ~0 and the loss ≈ ln 2
- `GRADIENT CHECK`
- per-step loss and rewards
- the post-training training-set pass
- `RUN RESULT`

**If this cell fails: STOP.** The failure is already logged.

In [ ]:
run([sys.executable, "training/train_dpo.py", "--sft-adapter-dir", SFT_ADAPTER_DIR, "--run-label", "baseline"])

## 10. Results (read from files written by the run)

In [ ]:
import glob, json, os
import pandas as pd
log = pd.read_csv("logs/experiment_log.csv")
row = log[log["method"] == "dpo"].iloc[-1]
display(row.to_frame("value"))
cols = ["total_steps", "final_train_loss", "ref_check_max_abs_reward",
        "train_set_reward_accuracy", "train_set_reward_margin", "train_set_chosen_reward", "train_set_rejected_reward",
        "peak_allocated_gb", "peak_reserved_gb", "peak_allocated_load_gb", "peak_reserved_load_gb",
        "peak_allocated_train_gb", "peak_reserved_train_gb", "peak_allocated_eval_gb", "peak_reserved_eval_gb",
        "alloc_retries", "cuda_ooms", "wall_clock_s", "avg_step_time_s",
        "sft_adapter_sha256", "preference_data_sha256", "policy_adapter_dtype", "adapter_path", "adapter_sha256"]
display(row[cols].to_frame("DPO (train_set_* = 150 TRAINING pairs, not held-out)"))
run_dir = sorted(glob.glob("outputs/dpo/dpo-baseline-*"))[-1]
manifest = json.load(open(os.path.join(run_dir, "run_manifest.json")))
hist = json.load(open(os.path.join(run_dir, "log_history.json")))
keys = ["step", "loss", "rewards/chosen", "rewards/rejected", "rewards/margins", "rewards/accuracies", "grad_norm", "learning_rate"]
display(pd.DataFrame([{k: h.get(k) for k in keys} for h in hist if "loss" in h and "step" in h]))
print("DPO ADAPTER:", manifest["adapter_path"], "sha256", manifest["adapter_sha256"])
!git status --short
!git diff --stat

## 11. Persist results
**(a)** Commit **only** `logs/experiment_log.csv` and push it, as for the SFT row.

**(b)** Copy the DPO run directory to Drive, with a SHA-256 check after the copy. The three-way evaluation needs it in a later session.

In [ ]:
from google.colab import drive
import shutil, hashlib
drive.mount("/content/drive")
dest_root = "/content/drive/MyDrive/week03_adapter_sprint/dpo"
os.makedirs(dest_root, exist_ok=True)
dest = os.path.join(dest_root, os.path.basename(run_dir))
if os.path.exists(dest):
    raise RuntimeError(f"STOP: {dest} already exists; not overwriting.")
shutil.copytree(run_dir, dest)
sha = hashlib.sha256(open(os.path.join(dest, "final_adapter", "adapter_model.safetensors"), "rb").read()).hexdigest()
print("copied to", dest, "| sha256 after copy:", sha, "MATCH" if sha == manifest["adapter_sha256"] else "MISMATCH - STOP")
assert sha == manifest["adapter_sha256"]

### Run conclusion (fill in by hand from the output above; do not guess)

| Check | Evidence | Result |
|---|---|---|
| Preflight on EXPECTED_COMMIT | §5 | |
| SFT adapter restored, SHA matches config + log | §6 | |
| Frozen instruction + preference data unchanged | §7 | |
| Dry-run precheck: 150 pairs, no truncation | §8 | |
| Policy == ref == SFT file (fp32) | §9 `POLICY / REFERENCE SETUP` | |
| Step-0 rewards ≈ 0, loss ≈ ln 2 | §9 / `ref_check_max_abs_reward` | |
| 10 optimizer steps; loss finite; `GRADIENT CHECK` passed | §9 / §10 | |
| Training-set reward accuracy / margin (labeled as such) | §10 | |
| Peak allocated / reserved (per phase), `alloc_retries` | §10 | |
| DPO adapter path + SHA-256; Drive copy verified | §10 / §11 | |
| Log row committed and pushed | §11 | |

**Interpretation limits (README §7.3 / §7.4):**
- **Training-set reward accuracy is not evidence of better address grounding.** The response-only digit-count shortcut (0.663) is a documented risk: DPO can raise reward accuracy by preferring digit-free outputs.
- Only the held-out three-way evaluation (base vs SFT vs DPO) can show whether fabrication fell **without** a rise in over-conservative nulls.